In [241]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [242]:
words = open('names.txt', 'r').read().splitlines()
len(words)

32033

In [243]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}

In [244]:
block_size = 3
X, Y = [], []

for w in words[:5]:
    print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        print(''.join(itos[i] for i in context), '---->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

emma
... ----> e
..e ----> m
.em ----> m
emm ----> a
mma ----> .
olivia
... ----> o
..o ----> l
.ol ----> i
oli ----> v
liv ----> i
ivi ----> a
via ----> .
ava
... ----> a
..a ----> v
.av ----> a
ava ----> .
isabella
... ----> i
..i ----> s
.is ----> a
isa ----> b
sab ----> e
abe ----> l
bel ----> l
ell ----> a
lla ----> .
sophia
... ----> s
..s ----> o
.so ----> p
sop ----> h
oph ----> i
phi ----> a
hia ----> .


In [245]:
X.shape

torch.Size([32, 3])

In [246]:
X

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [247]:
C = torch.randn(27,2)

In [248]:
for idx in range(len(C)):
    print(C[idx])
    print(itos[idx])

tensor([ 0.5453, -0.8242])
.
tensor([-1.6789,  0.6591])
a
tensor([ 0.6458, -1.8781])
b
tensor([-0.0495,  0.2458])
c
tensor([-0.2977,  2.0290])
d
tensor([-0.2257, -0.3726])
e
tensor([-1.0066, -1.8300])
f
tensor([0.8950, 0.9194])
g
tensor([ 0.5068, -0.4684])
h
tensor([-0.7269, -0.4308])
i
tensor([-0.0446, -0.9469])
j
tensor([ 0.6569, -0.2946])
k
tensor([0.9625, 1.9450])
l
tensor([-0.3959, -0.6839])
m
tensor([ 1.4084, -0.2575])
n
tensor([-0.0054,  1.7892])
o
tensor([-1.8783,  0.3437])
p
tensor([0.4248, 1.5151])
q
tensor([0.5009, 1.5856])
r
tensor([ 0.7923, -0.9882])
s
tensor([0.6928, 0.4587])
t
tensor([-0.4989,  0.9626])
u
tensor([-1.1191,  0.4655])
v
tensor([-0.0814,  0.5221])
w
tensor([-0.0711, -1.4392])
x
tensor([0.5643, 0.1611])
y
tensor([0.3206, 1.1391])
z


In [249]:
C[X]

tensor([[[ 0.5453, -0.8242],
         [ 0.5453, -0.8242],
         [ 0.5453, -0.8242]],

        [[ 0.5453, -0.8242],
         [ 0.5453, -0.8242],
         [-0.2257, -0.3726]],

        [[ 0.5453, -0.8242],
         [-0.2257, -0.3726],
         [-0.3959, -0.6839]],

        [[-0.2257, -0.3726],
         [-0.3959, -0.6839],
         [-0.3959, -0.6839]],

        [[-0.3959, -0.6839],
         [-0.3959, -0.6839],
         [-1.6789,  0.6591]],

        [[ 0.5453, -0.8242],
         [ 0.5453, -0.8242],
         [ 0.5453, -0.8242]],

        [[ 0.5453, -0.8242],
         [ 0.5453, -0.8242],
         [-0.0054,  1.7892]],

        [[ 0.5453, -0.8242],
         [-0.0054,  1.7892],
         [ 0.9625,  1.9450]],

        [[-0.0054,  1.7892],
         [ 0.9625,  1.9450],
         [-0.7269, -0.4308]],

        [[ 0.9625,  1.9450],
         [-0.7269, -0.4308],
         [-1.1191,  0.4655]],

        [[-0.7269, -0.4308],
         [-1.1191,  0.4655],
         [-0.7269, -0.4308]],

        [[-1.1191,  0

In [250]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

In [251]:
W1 = torch.rand(6,100)
b1 = torch.rand(100)

In [252]:
torch.cat([emb[:,0,:], emb[:,1,:], emb[:,2,:]], 1).shape

torch.Size([32, 6])

In [253]:
torch.cat(torch.unbind(emb, 1),1).shape

torch.Size([32, 6])

In [254]:
h = (emb.view(-1,6)) @ W1 + b1
#h = (emb.view(32,6)) @ W1 + b1
#h = (emb.view(emb.shape[0],6)) @ W1 + b1

W1.shape == 32,100

B1.shape == 100

32,100

1 ,100

broadcastable because each column has either equal values, 1, or DNE value

(emb.view(32,6)) @ W1 + b1 == (emb.view(-1,6)) @ W1 + b1 == (emb.view(emb.shape[0],6)) @ W1 + b1


In [255]:
h = torch.tanh(h)

In [256]:
W2 = torch.rand(100,27)
B2 = torch.rand(27)

In [257]:
logits = h @ W2 + B2

In [258]:
logits.shape

torch.Size([32, 27])

In [259]:
counts = logits.exp()

In [260]:
prob = counts/counts.sum(1,keepdims=True)

In [261]:
prob.shape

torch.Size([32, 27])

In [262]:
prob

tensor([[0.0519, 0.0063, 0.0257, 0.0442, 0.1443, 0.0025, 0.0177, 0.0103, 0.0056,
         0.0687, 0.0495, 0.0122, 0.0022, 0.0318, 0.1145, 0.0216, 0.0173, 0.0122,
         0.0548, 0.0014, 0.1931, 0.0041, 0.0078, 0.0074, 0.0029, 0.0298, 0.0599],
        [0.0159, 0.0053, 0.0262, 0.0371, 0.2682, 0.0074, 0.0236, 0.0105, 0.0022,
         0.0634, 0.0199, 0.0412, 0.0057, 0.0249, 0.2100, 0.0230, 0.0116, 0.0045,
         0.0177, 0.0043, 0.0915, 0.0040, 0.0073, 0.0188, 0.0026, 0.0204, 0.0327],
        [0.0042, 0.0049, 0.0395, 0.0100, 0.0967, 0.0090, 0.0051, 0.0273, 0.0001,
         0.0253, 0.0015, 0.1796, 0.0038, 0.0055, 0.4104, 0.0337, 0.0007, 0.0021,
         0.0079, 0.0122, 0.0067, 0.0027, 0.0098, 0.0441, 0.0003, 0.0526, 0.0042],
        [0.0003, 0.0017, 0.0094, 0.0020, 0.0187, 0.0014, 0.0017, 0.0045, 0.0000,
         0.0054, 0.0002, 0.0944, 0.0030, 0.0004, 0.7748, 0.0275, 0.0001, 0.0010,
         0.0008, 0.0115, 0.0005, 0.0007, 0.0067, 0.0064, 0.0001, 0.0261, 0.0006],
        [0.0001, 0.0027,

In [263]:
prob[0].sum()

tensor(1.0000)

In [264]:
loss = -prob[torch.arange(32), Y].log().mean()
loss

tensor(5.5185)

In [265]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

# Cleaned Version 1

In [304]:
block_size = 3
X, Y = [], []

for w in words:
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)

In [305]:
X.shape, Y.shape

(torch.Size([228146, 3]), torch.Size([228146]))

In [306]:
g = torch.Generator().manual_seed(2147483647)
C = torch.randn((27,2), generator=g, requires_grad=True)
W1 = torch.randn((6,100), generator=g, requires_grad=True)
b1 = torch.randn(100, generator=g, requires_grad=True)
W2 = torch.randn((100,27), generator=g, requires_grad=True)
b2 = torch.randn(27,generator=g, requires_grad=True)
parameters = [C, W1, b1, W2, b2]


In [307]:
sum(p.nelement() for p in parameters)

3481

In [310]:
for _ in range(100):
    #minibatch construct
    ix = torch.randint(0, X.shape[0],(32,))
    #forward pass
    emb = C[X[ix]]
    h = torch.tanh(emb.view(-1,6) @ W1 + b1)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits,Y[ix])

    #backward pass
    for p in parameters:
        p.grad = None
    loss.backward()
    print(loss.item())
    for p in parameters:
        p.data -= 0.1 * p.grad
        


9.669767379760742
9.674766540527344
7.740121841430664
9.717857360839844
8.712583541870117
9.611587524414062
7.020899772644043
6.949451923370361
8.531766891479492
8.952898979187012
7.595703601837158
8.418805122375488
6.778249740600586
6.205787181854248
6.588977813720703
7.151432037353516
8.028045654296875
6.383787631988525
6.687943458557129
4.739715099334717
5.179838180541992
6.559408187866211
4.928168773651123
5.07098913192749
5.216608047485352
4.794393539428711
5.861990928649902
6.058577537536621
6.245059013366699
5.2095112800598145
5.016918182373047
3.926831007003784
4.072482585906982
4.980342864990234
5.256199359893799
5.007011413574219
3.6779096126556396
6.385754585266113
5.301187515258789
5.391286373138428
4.357089519500732
4.295835971832275
4.855027198791504
4.213662147521973
4.171816825866699
5.753808975219727
4.506771564483643
3.651855945587158
4.13830041885376
4.370645046234131
3.7940125465393066
4.787753105163574
3.711397647857666
5.462850570678711
4.5257978439331055
3.342472